# Sse

> SSE parsing helpers.

In [ ]:
#| default_exp sse

## Imports

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass
from typing import AsyncIterator, Iterator, Optional
import httpx

In [ ]:
#| hide
from fastcore.test import *

## Purpose And Design

`sse.py` handles Server-Sent Events parsing in a provider-neutral way.

### Why this module exists

Streaming endpoints from different providers still rely on SSE framing rules. A dedicated parser keeps framing concerns separate from provider-specific event semantics.

### Architectural fit

- Upstream: line streams from HTTP responses.
- Downstream: `transport.stream_sse(_json)` and then provider normalizers.

### Key design choices

- Preserve SSE semantics (`event`, `data`, `id`, `retry`).
- Accept incremental/partial line sources robustly.
- Keep parser minimal and dependency-free.

### Reader outcome

After this notebook, you can reason about where a stream framing issue should be fixed (here) versus where provider event interpretation should be fixed (`normalize.py`).

### `SSEvent`

A parsed SSE event.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Core fields: `event: Optional[str]`, `data: str`, `id: Optional[str]`, `retry: Optional[int]`
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `transport`.
- Module downstream: `transport`.

In [ ]:
#| export
class SSEvent:
    "A parsed SSE event."
    event: Optional[str]
    data: str
    id: Optional[str] = None
    retry: Optional[int] = None

### `parse_sse_lines`

Parses synchronous SSE line streams into `SSEvent` objects.

**Why it exists**
- Provides a minimal framing parser reusable in tests and non-async contexts.

**Connections**
- Conceptual counterpart to `aiter_sse` used in async transport paths.

In [ ]:
#| export
def parse_sse_lines(lines: Iterator[str]) -> Iterator[SSEvent]:
    "Parse SSE line stream into events."
    event, data, event_id, retry = None, [], None, None
    for raw in lines:
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)

### `aiter_sse`

Async SSE parser over an `httpx` streamed response.

**Why it exists**
- Streaming providers emit event frames incrementally; this parser isolates framing from provider semantics.

**Connections**
- Called by `transport.stream_sse`, then decoded/normalized downstream.

In [ ]:
#| export
async def aiter_sse(response: httpx.Response) -> AsyncIterator[SSEvent]:
    "Async SSE parser from an httpx streamed response."
    event, data, event_id, retry = None, [], None, None
    async for raw in response.aiter_lines():
        line = raw.rstrip("\n")
        if not line:
            if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)
            event, data, event_id, retry = None, [], None, None
            continue
        if line.startswith(":"): continue
        field, _, value = line.partition(":")
        if value.startswith(" "): value = value[1:]
        if field == "event": event = value
        elif field == "data": data.append(value)
        elif field == "id": event_id = value
        elif field == "retry":
            try: retry = int(value)
            except ValueError: retry = None
    if data: yield SSEvent(event=event, data="\n".join(data), id=event_id, retry=retry)

## Quick Checks

In [ ]:
import fastllm.sse as m
for nm in ['SSEvent', 'parse_sse_lines', 'aiter_sse']: assert hasattr(m, nm), nm
from fastllm.sse import parse_sse_lines
evs = list(parse_sse_lines(iter(['data: {"a":1}\n','\n'])))
assert len(evs)==1 and evs[0].data=='{"a":1}'